# 上下文校正實驗（Workers AI，免費額度）

比較四種條件，看能不能讓字幕的教會專名更準、又不改壞其他字：

| 條件 | 做法 |
| --- | --- |
| 網站（基準） | 現在的網站：Workers AI Whisper 停頓切段 ≤10 秒＋固定替換＋同音校正 |
| A：Whisper 提示 | 辨識時把「最近出現的教會專名＋前一段字幕」放進 Whisper 的 `initial_prompt` |
| B：語言模型 | 網站字幕中「聽起來像專名」的段落，交給 Workers AI 語言模型只改錯字；再經安全檢查（同音、字數不變、不改已正確的專名） |
| A＋B | 兩者一起 |

**不需要 GPU**（預設 CPU 執行階段即可）。憑證沿用上次的 Colab 密鑰 `CF_ACCOUNT_ID`、`CF_API_TOKEN`。

### 順序（不要跳步）

1. `DATASET='life-study'`、`STAGE='pilot'`：4 篇生命讀經、每篇第 2–10 分鐘、兩個候選模型 → 把結果貼回對話，選定模型。
2. `STAGE='full'`：16 篇、只用選定的模型（會沿用試跑已完成的部分）。約需 2–3 天的免費額度，每天從第 1 格重新執行即可接續。
3. 開發集通過門檻後，`DATASET='meet'`、`CONFIRM_MEET=True` 只跑一次驗收。**MEET 是封存測試：不能依它的結果再改模型、提示或規則。**

### 額度

和網站共用每天 10,000 neurons（台灣早上 8 點重置）。本筆記本每次最多用 `NEURON_CAP`（預設 6,000），留給網站；
額度用完或達上限會自動暫停，已完成的請求都存在雲端硬碟。**請在沒有聚會的日子執行。**
執行前、後各看一次 Cloudflare 儀表板 → AI → Workers AI 的 neurons 用量，把兩個數字貼回對話（用來核對估算）。

生命讀經內容版權屬臺灣福音書房，只在你的雲端硬碟私人使用；這個實驗不訓練模型，也不上傳到 Git。

In [ ]:
# 第 1 格：設定
DATASET = 'life-study'        # 'life-study'（開發集）或 'meet'（封存測試，最後驗收才用）
STAGE = 'pilot'               # 'pilot'：4 篇＋兩個模型；'full'：16 篇＋選定的一個模型（MEET 不看這一項）
MODELS = '@cf/qwen/qwen3-30b-a3b-fp8,@cf/meta/llama-3.1-8b-instruct-fast'   # full／meet 時只留選定的一個
NEURON_CAP = 6000             # 這次最多用多少 neurons（估算）；其餘留給網站
WINDOW_START, WINDOW_SECONDS = 120, 480   # 生命讀經每篇只用第 2–10 分鐘
PILOT = '01,05,09,13'         # 試跑用的 4 篇
PRICES = {}                   # 第 5 格查不到價格時，依 Cloudflare 價格頁填：{'@cf/…': (每百萬輸入 token neurons, 每百萬輸出 token neurons)}
MEET_BASE_RUN = '20261111-082835'   # MEET 上次 Workers AI ≤10 秒的結果（網站基準直接沿用，不再花額度）
CONFIRM_MEET = False          # 驗收 MEET 時改成 True

In [ ]:
# 第 2 格：連接 Google Drive、找到資料
import os, subprocess
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive')

def find(name):
    direct = [ROOT/name, ROOT/'twgbr-archive'/name]
    for path in direct:
        if path.is_dir():
            return path
    found = [Path(d)/name for d, dirs, _ in os.walk(ROOT) if len(Path(d).relative_to(ROOT).parts) < 3 and name in dirs]
    if not found:
        raise RuntimeError(f'找不到 {name} 資料夾')
    return found[0]

models = [m.strip() for m in MODELS.split(',') if m.strip()]
if DATASET == 'life-study':
    DATA = find('life-study-dev-v1')
    ONLY = PILOT if STAGE == 'pilot' else ''
    if STAGE not in ('pilot', 'full'):
        raise RuntimeError("STAGE 只能是 'pilot' 或 'full'")
elif DATASET == 'meet':
    if not CONFIRM_MEET:
        raise RuntimeError('MEET 是封存測試，只在開發集通過、設定凍結後驗收一次；確定要跑請把 CONFIRM_MEET 改成 True')
    if len(models) != 1:
        raise RuntimeError('MEET 驗收只用選定的一個模型，請在 MODELS 只留一個')
    DATA = find('MEET'); ONLY = ''
    BASE_RAW = DATA/'eval-runs'/MEET_BASE_RUN
    if not any(BASE_RAW.glob('*.workers_ai_live_vad_10s.json')):
        raise RuntimeError(f'找不到 {BASE_RAW} 的 Workers AI ≤10 秒結果')
else:
    raise RuntimeError("DATASET 只能是 'life-study' 或 'meet'")
RUNS = DATA/'context-runs'
print('資料：', DATA, '｜', '試跑 ' + ONLY if ONLY else '全部')
print('模型：', *models)

In [ ]:
# 第 3 格：取得固定版本的程式與套件（約 1–2 分鐘）
import sys, base64, hashlib, io, json, tarfile, urllib.request
COMMIT = 'd6ad05142459fa2572b119a5235bc42151bed535'
REPO = Path('/content/Meeting-Live-Captions')
if not REPO.exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/timo893131/Meeting-Live-Captions.git', str(REPO)], check=True)
subprocess.run(['git', 'fetch', '-q', 'origin', COMMIT], cwd=REPO, check=True)
subprocess.run(['git', 'checkout', '-q', '--detach', COMMIT], cwd=REPO, check=True)
# faster-whisper 只用它的語音偵測（切段，和網站評估相同），不下載模型
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'faster-whisper==1.2.1', 'av==16.0.1', 'rapidfuzz==3.14.1',
                'opencc-python-reimplemented==0.1.7', 'nodejs-wheel-binaries==22.20.0'], check=True)
import nodejs_wheel
NODE = str(Path(nodejs_wheel.__file__).parent/'bin'/'node')
# 網站的 Worker 用 opencc-js 轉繁體：安裝 package-lock.json 釘住的版本並核對雜湊
lock = json.loads((REPO/'package-lock.json').read_text())['packages']['node_modules/opencc-js']
target = REPO/'node_modules/opencc-js'
if not (target/'package.json').exists():
    data = urllib.request.urlopen(lock['resolved'], timeout=120).read()
    if 'sha512-' + base64.b64encode(hashlib.sha512(data).digest()).decode() != lock['integrity']:
        raise RuntimeError('opencc-js 檔案雜湊不符，已停止')
    with tarfile.open(fileobj=io.BytesIO(data)) as archive_file:
        for member in archive_file.getmembers():
            if not member.isfile() or '..' in member.name.split('/'):
                continue
            path = target/member.name.split('/', 1)[1]
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_bytes(archive_file.extractfile(member).read())
print('程式版本', COMMIT[:12], '｜Node', subprocess.check_output([NODE, '--version'], text=True).strip())

In [ ]:
# 第 4 格：讀取 Colab 密鑰並確認權杖有效（不會印出權杖）
import urllib.error
from google.colab import userdata
env = dict(os.environ, CF_ACCOUNT_ID=userdata.get('CF_ACCOUNT_ID').strip(), CF_API_TOKEN=userdata.get('CF_API_TOKEN').strip())
API = f"https://api.cloudflare.com/client/v4/accounts/{env['CF_ACCOUNT_ID']}"
def cf_get(url):
    request = urllib.request.Request(url, headers={'Authorization': 'Bearer ' + env['CF_API_TOKEN']})
    with urllib.request.urlopen(request, timeout=60) as response:
        return json.loads(response.read())
try:
    status = cf_get(f'{API}/tokens/verify')['result']['status']
except urllib.error.HTTPError:
    status = cf_get('https://api.cloudflare.com/client/v4/user/tokens/verify')['result']['status']
print('API 權杖狀態：', status)
if status != 'active':
    raise RuntimeError('權杖不是 active，請重新建立')

In [ ]:
# 第 5 格：確認模型存在並取得價格（用來估算與控制 neurons）
NEURONS_PER_USD = 1000/0.011   # Workers AI：每 1,000 neurons 0.011 美元
catalog = {}
page = 1
while True:
    data = cf_get(f'{API}/ai/models/search?per_page=100&page={page}')
    for item in data.get('result', []):
        catalog[item.get('name')] = item
    if len(data.get('result', [])) < 100:
        break
    page += 1

def price_of(item):
    """(input, output) neurons per million tokens from the model's price property, if Cloudflare lists it."""
    found = {}
    for prop in item.get('properties', []):
        if prop.get('property_id') != 'price' or not isinstance(prop.get('value'), list):
            continue
        for entry in prop['value']:
            unit = str(entry.get('unit', '')).lower()
            if 'input' in unit:
                found['in'] = float(entry['price'])*NEURONS_PER_USD
            elif 'output' in unit:
                found['out'] = float(entry['price'])*NEURONS_PER_USD
    return (found['in'], found['out']) if len(found) == 2 else None

prices = {}
for model in models:
    if model not in catalog:
        raise RuntimeError(f'Workers AI 沒有 {model}；請從下面的清單改選')
    listed = price_of(catalog[model])
    chosen = PRICES.get(model) or listed
    if not chosen:
        raise RuntimeError(f'查不到 {model} 的價格：請到 Cloudflare 的 Workers AI 價格頁，把 neurons 填進第 1 格 PRICES')
    prices[model] = dict(input_neurons_per_m=chosen[0], output_neurons_per_m=chosen[1],
                         source='第 1 格手動填寫' if model in PRICES else 'Cloudflare 模型清單')
    print(f"{model}：每百萬 token 輸入 {chosen[0]:,.0f}／輸出 {chosen[1]:,.0f} neurons（{prices[model]['source']}）")
PRICE_FILE = Path('/content/context-prices.json')
PRICE_FILE.write_text(json.dumps(prices, indent=1))
text_models = sorted((n for n, i in catalog.items() if (i.get('task') or {}).get('name') == 'Text Generation'),
                     key=lambda n: (price_of(catalog[n]) or (9e9, 0))[0])
print('\n可用的文字模型（依輸入價格排序，前 15 個）：')
for name in text_models[:15]:
    p = price_of(catalog[name])
    print(' ', name, f'｜{p[0]:,.0f}／{p[1]:,.0f} neurons 每百萬 token' if p else '｜（清單未列價格）')

In [ ]:
# 第 6 格：切出和網站相同的段落（停頓切段 ≤10 秒；每次都重新計算，約 1–5 分鐘）
PREP = Path('/content/context-prep')/DATASET
command = [sys.executable, str(REPO/'scripts/context_experiment.py'), 'prepare', '--data', str(DATA), '--output', str(PREP)]
if DATASET == 'life-study':
    command += ['--window-start', str(WINDOW_START), '--window-seconds', str(WINDOW_SECONDS)]
if ONLY:
    command += ['--only', ONLY]
subprocess.run(command, check=True, cwd=REPO)

In [ ]:
# 第 7 格：執行（會沿用 context-runs 裡先前完成的部分；額度用完或達上限會暫停）
from datetime import datetime, timezone
earlier = sorted((p for p in RUNS.glob('*') if p.is_dir()), reverse=True) if RUNS.is_dir() else []
RUN = RUNS/datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
RUNS.mkdir(parents=True, exist_ok=True)
command = [NODE, str(REPO/'scripts/context-experiment.mjs'), '--prepared', str(PREP), '--output', str(RUN),
           '--conditions', 'base,prompt,llm,prompt_llm', '--models', ','.join(models), '--prices', str(PRICE_FILE),
           '--neuron-cap', str(NEURON_CAP)]
for folder in earlier:
    command += ['--reuse', str(folder)]
if DATASET == 'meet':
    command += ['--base-raw', str(BASE_RAW)]
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=REPO, env=env)
for line in process.stdout:
    print(line, end='', flush=True)
code = process.wait()
if code == 3:
    print('\n⏸ 已暫停。明天台灣早上 8 點後從第 1 格依序再執行一次（設定不要改），就會從暫停處繼續。第 8 格可以先看已完成部分的結果。')
elif code:
    raise RuntimeError('執行失敗，請看上面的錯誤訊息')

In [ ]:
# 第 8 格：計分（請把印出的表格貼回對話）
command = [sys.executable, str(REPO/'scripts/context_experiment.py'), 'score', '--data', str(DATA), '--prepared', str(PREP), '--run', str(RUN)]
if DATASET == 'life-study':
    command.append('--development')
subprocess.run(command, check=True, cwd=REPO)
print('\n結果也存在：', RUN/'context-summary.md')

## 請貼回對話

1. 第 8 格印出的兩張表與「門檻」各行（開發集的「修改明細」也請貼，或告訴我檔案位置；MEET 不會列明細）。
2. 執行前、後 Cloudflare 儀表板的 neurons 用量。

怎麼看：**教會專名找到／多出**是主要指標（生命讀經的文字不是逐字稿，所以修好／改壞字數與 CER 只供參考）；
「未經安全檢查」那一行是語言模型的原始建議，用來看安全檢查擋掉了多少亂改；
**提示被照抄**是 Whisper 把提示裡的前一段字幕又寫一次的段數（應為 0）；
語言模型的延遲是 Colab 到 Cloudflare 的往返，上線後由 Worker 直接呼叫，通常更短。